## 실습 2: 메모리를 추가하여 에이전트 개인화하기

### 개요

실습 1에서는 로컬 세션의 단일 사용자에게 잘 작동하는 고객 지원 에이전트를 구축했습니다. 하지만 실제 고객 지원 환경에서는 로컬 환경에서 실행되는 단일 사용자를 넘어 확장할 수 있어야 합니다.

**프로덕션에서 에이전트**를 실행하려면 다음 기능이 필요합니다.
- **다중 사용자 지원**: 수천 명의 고객을 동시에 처리
- **영구 스토리지**: 세션 수명 주기가 끝난 후에도 대화 저장
- **장기 학습**: 고객 선호도 및 행동 패턴 추출
- **세션 간 연속성**: 서로 다른 상호 작용에서도 고객 정보 기억

**워크숍 진행 과정:**
- **실습 1 (완료)**: 에이전트 프로토타입 만들기 - 실제로 작동하는 고객 지원 에이전트 구축
- **실습 2 (현재)**: 메모리로 기능 강화 - 대화 컨텍스트 및 개인화 추가
- **실습 3**: Gateway 및 Identity로 확장 - 여러 에이전트에서 도구를 안전하게 공유
- **실습 4**: 프로덕션에 배포 - 관찰 기능과 함께 AgentCore Runtime 사용
- **실습 5**: 사용자 인터페이스 구축 - 고객용 애플리케이션 만들기

이 실습에서는 누락된 영속성 및 학습 계층을 추가하여 대화를 금세 잊는 에이전트를 스마트한 개인화 어시스턴트로 전환합니다.

메모리는 지능의 핵심 구성 요소입니다. Large Language Models (LLMs)은 뛰어난 기능을 갖추고 있지만 대화 간 영구 메모리가 없습니다. [Amazon Bedrock AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-getting-started.html)는 AI 에이전트가 시간의 흐름에 따라 컨텍스트를 유지하고, 중요한 사실을 기억하며, 일관된 개인화 경험을 제공할 수 있게 하는 관리형 서비스로 이러한 한계를 해결합니다.

AgentCore Memory는 다음 두 수준으로 작동합니다.
- **단기 메모리(Short-Term Memory)**: 단일 상호 작용 또는 밀접하게 관련된 세션 내에서 연속성을 제공하는 즉각적인 대화 컨텍스트 및 세션 기반 정보입니다.
- **장기 메모리(Long-Term Memory)**: 여러 대화에서 추출하여 저장한 영구 정보로, 시간이 지나도 개인화 경험을 제공할 수 있도록 사실, 선호도, 요약 등을 포함합니다.

### 실습 2 아키텍처
<div style="text-align:left">
    <img src="images/architecture_lab2_memory.png" width="75%"/>
</div>

*영구적인 단기 및 장기 메모리 기능을 갖춘 다중 사용자 에이전트입니다.*

### 사전 요구 사항

* 적절한 권한이 있는 **AWS 계정**
* 로컬에 설치된 **Python 3.10 이상**
* 자격 증명이 설정된 **AWS CLI**
* 다음 셀에서 설치할 **Google ADK** 및 기타 라이브러리
* AWS 워크숍 계정에는 다음 리소스가 미리 생성되어 있습니다.
    - AWS Lambda 함수
    - Amazon Bedrock Knowledge Base


### 단계 1: 라이브러리 가져오기

AgentCore Memory용 라이브러리를 가져옵니다. 여기서는 [Amazon Bedrock AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html) SDK를 사용합니다.

또한 에이전트를 구축할 Google ADK 구성 요소를 가져오고, **LiteLLM**을 사용하여 **Amazon Bedrock** 모델을 호출합니다.

> **참고:** Google ADK는 비동기 패턴을 사용합니다. Jupyter Notebook에서는 이미 asyncio 이벤트 루프가 실행 중이므로 최상위 수준에서 `await`를 직접 사용할 수 있습니다.

> **참고:** Google ADK의 `LiteLlm`을 사용하여 요청을 Amazon Bedrock 모델로 라우팅합니다. 따라서 Gemini API key가 필요하지 않으며, 대신 AWS 자격 증명을 사용합니다.

In [ ]:
import uuid
import time

from lab_helpers.utils import suppress_warnings

suppress_warnings()

import boto3
from boto3.session import Session

from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

from lab_helpers.utils import put_ssm_parameter

# boto 세션 및 리전 가져오기
boto_session = Session()
REGION = boto_session.region_name
print(f"Region: {REGION}")

### 단계 2: Bedrock AgentCore Memory 리소스 생성

Amazon Bedrock AgentCore Memory는 AI 에이전트에 영구 메모리를 제공하는 완전 관리형 서비스입니다. 다음 두 가지 전략을 사용하여 Memory 리소스를 생성합니다.

1. **User Preference Strategy**: 고객 선호도 및 행동 패턴 수집
2. **Semantic Strategy**: 벡터 임베딩을 사용하여 대화의 사실 정보 저장

각 전략은 체계적인 검색을 위해 고유한 네임스페이스를 사용합니다.

In [ ]:
memory_name = "CustomerSupportMemory"

memory_manager = MemoryManager(region_name=REGION)
memory = memory_manager.get_or_create_memory(
    name=memory_name,
    strategies=[
        {
            StrategyType.USER_PREFERENCE.value: {
                "name": "CustomerPreferences",
                "description": "Captures customer preferences and behavior",
                "namespaces": ["support/customer/{actorId}/preferences/"],
            }
        },
        {
            StrategyType.SEMANTIC.value: {
                "name": "CustomerSupportSemantic",
                "description": "Stores facts from conversations",
                "namespaces": ["support/customer/{actorId}/semantic/"],
            }
        },
    ],
)
memory_id = memory["id"]
put_ssm_parameter("/app/customersupport/agentcore/memory_id", memory_id)

In [ ]:
if memory_id:
    print("\u2705 AgentCore Memory created successfully!")
    print(f"Memory ID: {memory_id}")
else:
    print("Memory resource not created. Try Again !")

## 단계 3: 이전 고객 상호 작용 시드 데이터 추가

**Memory에 시드 데이터를 추가하는 이유는 무엇인가요?**

프로덕션에서는 고객과 상호 작용하면서 에이전트에 메모리가 자연스럽게 축적됩니다. 하지만 이 실습에서는 실제 대화를 기다리지 않고 Long-Term Memory (LTM)의 작동 방식을 살펴볼 수 있도록 과거 대화를 시드 데이터로 추가합니다.

**Memory 처리 방식:**
1. `create_event`가 상호 작용을 **Short-Term Memory** (STM)에 즉시 저장합니다.
2. **Long-Term Memory** 전략이 STM을 비동기 방식으로 처리합니다.
3. LTM이 나중에 검색할 패턴, 선호도, 사실을 추출합니다.

이 과정을 확인할 수 있도록 일부 고객 기록을 시드 데이터로 추가합니다.

In [ ]:
from lab_helpers.lab2_memory import ACTOR_ID


# 이전 고객 상호 작용을 시드 데이터로 추가
previous_interactions = [
    ("I'm having issues with my MacBook Pro overheating during video editing.", "USER"),
    (
        "I can help with that thermal issue. For video editing workloads, let's check your Activity Monitor and adjust performance settings. Your MacBook Pro order #MB-78432 is still under warranty.",
        "ASSISTANT",
    ),
    (
        "What's the return policy on gaming headphones? I need low latency for competitive FPS games",
        "USER",
    ),
    (
        "For gaming headphones, you have 30 days to return. Since you're into competitive FPS, I'd recommend checking the audio latency specs - most gaming models have <40ms latency.",
        "ASSISTANT",
    ),
    (
        "I need a laptop under $1200 for programming. Prefer 16GB RAM minimum and good Linux compatibility. I like ThinkPad models.",
        "USER",
    ),
    (
        "Perfect! For development work, I'd suggest looking at our ThinkPad E series or Dell XPS models. Both have excellent Linux support and 16GB RAM options within your budget.",
        "ASSISTANT",
    ),
]

# 이전 상호 작용 저장
if memory_id:
    try:
        memory_client = MemoryClient(region_name=REGION)
        memory_client.create_event(
            memory_id=memory_id,
            actor_id=ACTOR_ID,
            session_id="previous_session",
            messages=previous_interactions,
        )
        print("\u2705 Seeded customer history successfully")
        print("\U0001f4dd Interactions saved to Short-Term Memory")
        print("\u23f3 Long-Term Memory processing will begin automatically...")
    except Exception as e:
        print(f"\u26a0\ufe0f Error seeding history: {e}")

### Memory 처리 방식 이해

`create_event`로 이벤트를 생성하면 AgentCore Memory가 데이터를 두 단계로 처리합니다.

1. **즉시 처리**: 메시지를 Short-Term Memory (STM)에 저장
2. **비동기 처리**: STM을 Long-Term Memory (LTM) 전략으로 처리

시스템에서 다음 작업을 수행하므로 LTM 처리에는 일반적으로 20~30초가 걸립니다.
- 대화 패턴 분석
- 고객 선호도 및 행동 추출
- 사실 정보에 대한 시맨틱 임베딩 생성
- 효율적인 검색을 위해 네임스페이스별로 Memory 구성

고객 선호도를 검색하여 Long-Term Memory 처리가 완료되었는지 확인합니다.

In [ ]:
# Long-Term Memory 처리가 완료될 때까지 대기
print("\U0001f50d Checking for processed Long-Term Memories...")
retries = 0
max_retries = 6  # 1분 대기

while retries < max_retries:
    memories = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace=f"support/customer/{ACTOR_ID}/preferences/",
        query="can you summarize the support issue",
    )

    if memories:
        print(f"\u2705 Found {len(memories)} preference memories after {retries * 10} seconds!")
        break

    retries += 1
    if retries < max_retries:
        print(f"\u23f3 Still processing... waiting 10 more seconds (attempt {retries}/{max_retries})")
        time.sleep(10)
    else:
        print("\u26a0\ufe0f Memory processing is taking longer than expected. This can happen with overloading..")
        break

print("\U0001f3af AgentCore Memory automatically extracted these customer preferences from our seeded conversations:")
print("=" * 80)

for i, memory in enumerate(memories, 1):
    if isinstance(memory, dict):
        content = memory.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
            print(f"  {i}. {text}")

### Semantic Memory 살펴보기

Semantic Memory는 벡터 임베딩을 사용하여 대화의 사실 정보를 저장합니다. 이를 통해 관련 사실과 컨텍스트를 유사도 기반으로 검색할 수 있습니다.

In [ ]:
import time

# Semantic Memory(사실 정보) 검색
while True:
    semantic_memories = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace=f"support/customer/{ACTOR_ID}/semantic/",
        query="information on the technical support issue",
    )
    print("\U0001f9e0 AgentCore Memory identified these factual details from conversations:")
    print("=" * 80)
    if semantic_memories:
        break
    time.sleep(10)
for i, memory in enumerate(semantic_memories, 1):
    if isinstance(memory, dict):
        content = memory.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
            print(f"  {i}. {text}")

## 단계 4: Memory를 사용하는 고객 지원 에이전트 생성

다음으로 실습 1과 같은 방식으로 고객 지원 에이전트를 구현하되, 이번에는 AgentCore Memory를 통합합니다.

Google ADK에는 Strands와 같은 기본 제공 `AgentCoreMemorySessionManager`가 없으므로 Memory 통합을 명시적으로 구현합니다.

1. **각 질의 전**: Memory에서 관련 고객 컨텍스트 검색(preferences + semantic)
2. **컨텍스트 주입**: LLM이 개인화 컨텍스트를 활용할 수 있도록 검색한 메모리를 사용자 질의 앞에 추가
3. **각 응답 후**: 나중에 사용할 수 있도록 상호 작용(질의 + 응답)을 Memory에 다시 저장

이 패턴을 사용하면 검색 및 저장 로직을 완전히 제어하면서 동일한 Memory 강화 동작을 구현할 수 있습니다.

> **참고:** `LlmAgent`와 `LiteLlm`을 사용하여 Google ADK를 통해 Amazon Bedrock 모델(Claude)을 호출합니다. 따라서 LLM 호출에는 Gemini API key가 아닌 AWS 자격 증명이 사용됩니다.

> **참고:** 실습 1의 도구(get_return_policy, get_product_info, web_search, get_technical_support)를 그대로 재사용합니다.

In [ ]:
# ============================================================
# 실습 1과 동일한 도구 정의
# ============================================================


def get_return_policy(product_category: str) -> str:
    """
    Get return policy information for a specific product category.

    Args:
        product_category: Electronics category (e.g., 'smartphones', 'laptops', 'accessories')

    Returns:
        Formatted return policy details including timeframes and conditions
    """
    return_policies = {
        "smartphones": {
            "window": "30 days",
            "condition": "Original packaging, no physical damage, factory reset required",
            "process": "Online RMA portal or technical support",
            "refund_time": "5-7 business days after inspection",
            "shipping": "Free return shipping, prepaid label provided",
            "warranty": "1-year manufacturer warranty included",
        },
        "laptops": {
            "window": "30 days",
            "condition": "Original packaging, all accessories, no software modifications",
            "process": "Technical support verification required before return",
            "refund_time": "7-10 business days after inspection",
            "shipping": "Free return shipping with original packaging",
            "warranty": "1-year manufacturer warranty, extended options available",
        },
        "accessories": {
            "window": "30 days",
            "condition": "Unopened packaging preferred, all components included",
            "process": "Online return portal",
            "refund_time": "3-5 business days after receipt",
            "shipping": "Customer pays return shipping under $50",
            "warranty": "90-day manufacturer warranty",
        },
    }
    default_policy = {
        "window": "30 days",
        "condition": "Original condition with all included components",
        "process": "Contact technical support",
        "refund_time": "5-7 business days after inspection",
        "shipping": "Return shipping policies vary",
        "warranty": "Standard manufacturer warranty applies",
    }
    policy = return_policies.get(product_category.lower(), default_policy)
    return (
        f"Return Policy - {product_category.title()}:\n\n"
        f"\u2022 Return window: {policy['window']} from delivery\n"
        f"\u2022 Condition: {policy['condition']}\n"
        f"\u2022 Process: {policy['process']}\n"
        f"\u2022 Refund timeline: {policy['refund_time']}\n"
        f"\u2022 Shipping: {policy['shipping']}\n"
        f"\u2022 Warranty: {policy['warranty']}"
    )

In [ ]:
def get_product_info(product_type: str) -> str:
    """
    Get detailed technical specifications and information for electronics products.

    Args:
        product_type: Electronics product type (e.g., 'laptops', 'smartphones', 'headphones', 'monitors')
    Returns:
        Formatted product information including warranty, features, and policies
    """
    products = {
        "laptops": {
            "warranty": "1-year standard, 3-year extended available",
            "specs": "Intel/AMD processors, 8-64GB RAM, SSD storage",
            "features": "Backlit keyboards, fingerprint readers, Thunderbolt ports",
            "compatibility": "Windows, Linux, macOS (Apple only)",
            "support": "24/7 technical support, on-site repair options",
        },
        "smartphones": {
            "warranty": "1-year manufacturer, 2-year extended",
            "specs": "Latest processors, 6-12GB RAM, 128GB-1TB storage",
            "features": "5G capable, water resistant, wireless charging",
            "compatibility": "iOS or Android ecosystem",
            "support": "In-store and mail-in repair services",
        },
        "headphones": {
            "warranty": "1-year standard warranty",
            "specs": "Bluetooth 5.0+, ANC, 20-40hr battery",
            "features": "Active noise cancellation, transparency mode, multipoint",
            "compatibility": "Universal Bluetooth, some with proprietary apps",
            "support": "Replacement program for defective units",
        },
        "monitors": {
            "warranty": "3-year standard, zero dead pixel guarantee",
            "specs": "4K/1440p resolution, 60-240Hz refresh rate",
            "features": "HDR support, high refresh rates, adjustable stands",
            "compatibility": "HDMI, DisplayPort, USB-C inputs",
            "support": "Color calibration and technical support",
        },
    }
    product = products.get(product_type.lower())
    if not product:
        return f"Technical specifications for {product_type} not available. Please contact our technical support team for detailed product information and compatibility requirements."
    return (
        f"Technical Information - {product_type.title()}:\n\n"
        f"\u2022 Warranty: {product['warranty']}\n"
        f"\u2022 Specifications: {product['specs']}\n"
        f"\u2022 Key Features: {product['features']}\n"
        f"\u2022 Compatibility: {product['compatibility']}\n"
        f"\u2022 Support: {product['support']}"
    )


def web_search(keywords: str) -> str:
    """Search the web for updated information.

    Args:
        keywords: The search query keywords.
    Returns:
        List of dictionaries with search results.
    """
    try:
        results = DDGS().text(keywords, region="us-en", max_results=5)
        return str(results) if results else "No results found."
    except RatelimitException:
        return "Rate limit reached. Please try again later."
    except DDGSException as e:
        return f"Search error: {e}"
    except Exception as e:
        return f"Search error: {str(e)}"


def get_technical_support(issue_description: str) -> str:
    """Search the technical support knowledge base for troubleshooting help, setup guides, and maintenance tips.

    Args:
        issue_description: Description of the technical issue or question the customer needs help with.
    Returns:
        Relevant technical support documentation and troubleshooting steps.
    """
    try:
        ssm = boto3.client("ssm")
        account_id = boto3.client("sts").get_caller_identity()["Account"]
        region = boto3.Session().region_name
        kb_id = ssm.get_parameter(Name=f"/{account_id}-{region}/kb/knowledge-base-id")["Parameter"]["Value"]
        bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=region)
        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=kb_id,
            retrievalQuery={"text": issue_description},
            retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
        )
        results = response.get("retrievalResults", [])
        if not results:
            return "No relevant technical support documentation found for this issue."
        formatted_results = []
        for i, result in enumerate(results, 1):
            text = result.get("content", {}).get("text", "")
            score = result.get("score", 0)
            if score >= 0.4:
                formatted_results.append(f"--- Result {i} (relevance: {score:.2f}) ---\n{text}")
        if not formatted_results:
            return "No sufficiently relevant technical support documentation found."
        return "\n\n".join(formatted_results)
    except Exception as e:
        return f"Unable to access technical support documentation. Error: {str(e)}"


print("\u2705 All tools defined")

In [ ]:
APP_NAME = "customer_support_agent"
USER_ID = "user1234"

SYSTEM_PROMPT = """You are a helpful and professional customer support assistant for an electronics e-commerce company.
Your role is to:
- Provide accurate information using the tools available to you
- Support the customer with technical information and product specifications, and maintenance questions
- Be friendly, patient, and understanding with customers
- Always offer additional help after answering questions
- If you can't help with something, direct customers to the appropriate contact

You have access to the following tools:
1. get_return_policy() - For warranty and return policy questions
2. get_product_info() - To get information about a specific product
3. web_search() - To access current technical documentation, or for updated information.
4. get_technical_support() - For troubleshooting issues, setup guides, maintenance tips, and detailed technical assistance
For any technical problems, setup questions, or maintenance concerns, always use the get_technical_support() tool.

Always use the appropriate tool to get accurate, up-to-date information rather than making assumptions."""

# LiteLLM으로 Amazon Bedrock 모델을 호출하는 Google ADK 에이전트 생성
# LLM 호출은 AWS 자격 증명을 통해 라우팅되므로 Gemini API key가 필요하지 않습니다.
agent = LlmAgent(
    model=LiteLlm(model="bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    name="customer_support_agent",
    description="A customer support agent for an electronics e-commerce company.",
    instruction=SYSTEM_PROMPT,
    tools=[
        get_product_info,
        get_return_policy,
        web_search,
        get_technical_support,
    ],
)

print("\u2705 Customer Support Agent created with Amazon Bedrock model via LiteLLM")

In [ ]:
# Memory 강화 헬퍼: 컨텍스트 검색 -> 에이전트 실행 -> 상호 작용 저장
async def call_agent_with_memory(query: str, session_id: str = None):
    """Memory 컨텍스트를 주입하여 에이전트에 질의를 보냅니다."""
    if session_id is None:
        session_id = str(uuid.uuid4())

    # --- 1. Memory에서 고객 컨텍스트 검색 ---
    all_context = []
    namespaces = {
        "preferences": f"support/customer/{ACTOR_ID}/preferences/",
        "semantic": f"support/customer/{ACTOR_ID}/semantic/",
    }
    for context_type, namespace in namespaces.items():
        try:
            memories = memory_client.retrieve_memories(
                memory_id=memory_id,
                namespace=namespace,
                query=query,
                top_k=3,
            )
            for mem in memories:
                if isinstance(mem, dict):
                    text = mem.get("content", {}).get("text", "").strip()
                    if text:
                        all_context.append(f"[{context_type.upper()}] {text}")
        except Exception as e:
            print(f"Warning: Could not retrieve {context_type} memories: {e}")

    # --- 2. 컨텍스트로 보강된 질의 구성 ---
    if all_context:
        context_text = "\n".join(all_context)
        enriched_query = f"Customer Context:\n{context_text}\n\n{query}"
        print(f"\U0001f4cb Retrieved {len(all_context)} memory items for context")
    else:
        enriched_query = query
        print("No prior memory context found")

    # --- 3. ADK 에이전트 실행 ---
    session_service = InMemorySessionService()
    await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=session_id)
    runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)
    content = types.Content(role="user", parts=[types.Part(text=enriched_query)])

    final_response = ""
    async for event in runner.run_async(user_id=USER_ID, session_id=session_id, new_message=content):
        if event.is_final_response():
            final_response = event.content.parts[0].text
            print("\nAgent Response:\n", final_response)

    # --- 4. 상호 작용을 Memory에 저장 ---
    if final_response:
        try:
            memory_client.create_event(
                memory_id=memory_id,
                actor_id=ACTOR_ID,
                session_id=session_id,
                messages=[
                    (query, "USER"),
                    (final_response, "ASSISTANT"),
                ],
            )
            print("\U0001f4be Interaction saved to memory")
        except Exception as e:
            print(f"Warning: Could not save to memory: {e}")

    return final_response


print("\u2705 Memory-enhanced agent helper ready")

## 단계 5: 개인화 에이전트 테스트

Memory로 강화된 에이전트를 테스트합니다. 에이전트가 고객의 과거 선호도를 활용하여 개인화된 추천을 제공하는 방식을 확인해 보세요.

에이전트는 다음 작업을 자동으로 수행합니다.
1. Memory에서 관련 고객 컨텍스트 검색
2. 해당 컨텍스트를 사용하여 응답 개인화
3. 나중에 사용할 수 있도록 새로운 상호 작용 저장

In [ ]:
print("\U0001f3a7 Testing headphone recommendation with customer memory...\n\n")
response1 = await call_agent_with_memory("Which headphones would you recommend?")

In [ ]:
print("\n\U0001f4bb Testing laptop preference recall...\n\n")
response2 = await call_agent_with_memory("What is my preferred laptop brand and requirements?")

에이전트가 다음 정보를 어떻게 기억하는지 확인해 보세요.

- 게임 관련 선호도(저지연 헤드폰)
- 노트북 선호도(ThinkPad, 16GB RAM, Linux 호환성)
- 예산 제약(노트북 구매에 $1200)
- 이전 기술 문제(MacBook 과열)

이것이 바로 AgentCore Memory가 제공하는 영구적이고 개인화된 고객 경험입니다.

## 축하합니다! 🎉

**실습 2: 고객 지원 에이전트에 메모리 추가**를 성공적으로 완료했습니다.

### 이번 실습에서 완료한 내용

- Amazon Bedrock AgentCore Memory를 사용하여 서버리스 관리형 메모리 생성
- User-Preferences 및 Semantic(사실) 정보를 저장하는 장기 메모리 구현
- Google ADK에서 명시적인 메모리 검색 및 저장을 구현하여 AgentCore Memory와 고객 지원 에이전트 통합
- LiteLLM을 사용하여 Google ADK를 통해 Amazon Bedrock 모델(Claude) 호출

##### 다음 실습: [실습 3 - Gateway 및 Identity로 확장하기 \u2192](lab-03-agentcore-gateway.ipynb)

## 리소스
- [Amazon Bedrock Agent Core Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html)
- [Amazon Bedrock AgentCore Memory Deep Dive blog](https://aws.amazon.com/blogs/machine-learning/amazon-bedrock-agentcore-memory-building-context-aware-agents/)
- [Google ADK Documentation](https://google.github.io/adk-docs/)
- [Google ADK LiteLLM Integration](https://google.github.io/adk-docs/agents/models/litellm/)
- [AgentCore with Google ADK Integration](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/using-any-agent-framework.html#agent-runtime-frameworks-google-adk)